# JobLens AI: Job Matching and Skill Extraction Pipeline

This notebook combines resume–job matching evaluation with job-description skill extraction. Run the sections in order; each section can also be used independently after its dependencies are installed.

## Setup

In [ ]:
# Run this once in Google Colab.
# !pip install -q sentence-transformers datasets kagglehub pandas scikit-learn transformers torch spacy matplotlib openai python-dotenv
# !python -m spacy download en_core_web_sm -q

## Part 1: Resume–Job Matching

In [ ]:
from __future__ import annotations

import gc
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from datasets import load_dataset
from kagglehub import KaggleDatasetAdapter
import kagglehub
from sentence_transformers import (
    InputExample, SentenceTransformer, evaluation, losses
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    base_model_name: str = 'all-MiniLM-L6-v2'
    drive_model_path: str = '/content/drive/MyDrive/joblens-finetuned-sbert'
    local_model_path: str = 'joblens-finetuned-sbert'
    max_jobs: int = 50
    max_resumes: int = 100
    top_k: int = 3
    batch_size: int = 16
    epochs: int = 3
    device: str = 'cpu'
    esco_mapping_min_score: float = 0.08
    esco_mapping_chunk_size: int = 512
    test_fraction: float = 0.5
    n_bootstrap: int = 2000


CONFIG = PipelineConfig()


### Data preparation functions

These functions load and clean the data, then construct ESCO-aligned weak-supervision labels without a hand-written category-to-title dictionary. `prepare_evaluation_data` splits jobs (and the resumes they resolve to) into a **train** pool and a disjoint **test** pool before any fine-tuning happens, so the fine-tuned model is never evaluated on job/resume pairs it saw during training. `export_esco_mapping_sample_for_annotation` and `evaluate_weak_label_agreement` let you measure how accurate the ESCO weak-supervision labels themselves are, since every downstream metric depends on them being correct.


In [ ]:
def load_resume_dataset() -> pd.DataFrame:
    """Convert Hugging Face Resume Atlas into the standard resume DataFrame."""
    raw_resumes = load_dataset('ahmedheakl/resume-atlas')['train'].to_pandas()
    return pd.DataFrame({
        'id': [f'R{i}' for i in range(len(raw_resumes))],
        'category': raw_resumes['Category'].fillna('').str.strip(),
        'resume_text': raw_resumes['Text'].fillna('').astype(str),
    })


def load_job_postings() -> pd.DataFrame:
    """Convert Kaggle LinkedIn postings into the standard job DataFrame."""
    raw_jobs = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, 'arshkon/linkedin-job-postings', 'postings.csv'
    )
    return pd.DataFrame({
        'id': [f'J{i}' for i in range(len(raw_jobs))],
        'title': raw_jobs['title'].fillna('').astype(str),
        'job_description': raw_jobs['description'].fillna('').astype(str),
    })


def _esco_data_path(filename: str) -> Path:
    candidates = (Path('/colab/data'), Path('colab/data'), Path('data'))
    path = next((base / filename for base in candidates if (base / filename).exists()), None)
    if path is None:
        raise FileNotFoundError(f'ESCO data file not found: {filename}')
    return path


def load_esco_occupation_catalog() -> pd.DataFrame:
    """Load ESCO occupations as the data-driven weak-supervision reference."""
    occupations = pd.read_csv(
        _esco_data_path('occupations_en.csv'),
        usecols=['conceptUri', 'preferredLabel', 'altLabels', 'description'],
    ).fillna('')
    occupations['occupation_text'] = (
        occupations['preferredLabel'] + ' ' + occupations['altLabels'].str.replace('\n', ' ', regex=False)
        + ' ' + occupations['description'].str.slice(0, 600)
    ).str.lower()
    return occupations.rename(columns={'conceptUri': 'esco_occupation_uri'})


def map_texts_to_esco_occupations(texts: pd.Series, occupation_catalog: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    """Independently map job/resume text to ESCO occupation URIs with character TF-IDF."""
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1)
    catalog_matrix = vectorizer.fit_transform(occupation_catalog['occupation_text'])
    text_matrix = vectorizer.transform(texts.fillna('').astype(str).str.lower())
    mapped_rows = []
    for start in range(0, text_matrix.shape[0], config.esco_mapping_chunk_size):
        scores = (text_matrix[start:start + config.esco_mapping_chunk_size] @ catalog_matrix.T).toarray()
        best_indices = np.argmax(scores, axis=1)
        best_scores = scores[np.arange(len(best_indices)), best_indices]
        mapped_rows.extend(zip(
            occupation_catalog.iloc[best_indices]['esco_occupation_uri'],
            occupation_catalog.iloc[best_indices]['preferredLabel'],
            best_scores,
        ))
    return pd.DataFrame(mapped_rows, columns=['esco_occupation_uri', 'esco_occupation_label', 'esco_mapping_score'])


def _build_job_resume_split(
    job_ids: set, mapped_jobs: pd.DataFrame, available_resumes: pd.DataFrame,
    resume_pool_seed_offset: int, max_jobs: int, max_resumes: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Tag one job/resume split with ESCO-aligned correct-resume labels from a resume pool."""
    split_jobs = mapped_jobs[mapped_jobs['id'].isin(job_ids)]
    if available_resumes.empty:
        return split_jobs.iloc[0:0], available_resumes.iloc[0:0]
    occupation_to_resume_id = (
        available_resumes.groupby('esco_occupation_uri')
        .sample(n=1, random_state=RANDOM_SEED + resume_pool_seed_offset)
        .set_index('esco_occupation_uri')['id'].to_dict()
    )
    split_jobs = split_jobs.copy()
    split_jobs['correct_resume_id'] = split_jobs['esco_occupation_uri'].map(occupation_to_resume_id)
    split_jobs = (split_jobs.dropna(subset=['correct_resume_id'])
                  .sample(n=min(max_jobs, len(split_jobs.dropna(subset=['correct_resume_id']))), random_state=RANDOM_SEED)
                  .reset_index(drop=True))
    if split_jobs.empty:
        return split_jobs, available_resumes.iloc[0:0]
    required_ids = set(split_jobs['correct_resume_id'])
    required_resumes = available_resumes[available_resumes['id'].isin(required_ids)]
    optional_resumes = available_resumes[~available_resumes['id'].isin(required_ids)]
    remaining_count = max(0, max_resumes - len(required_resumes))
    sampled_optional = optional_resumes.sample(n=min(remaining_count, len(optional_resumes)), random_state=RANDOM_SEED + resume_pool_seed_offset)
    split_resumes = pd.concat([required_resumes, sampled_optional], ignore_index=True)
    return split_jobs, split_resumes


def prepare_evaluation_data(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build job-disjoint and resume-disjoint train/test pools aligned to ESCO occupations.

    Jobs are split into train/test *before* any resume is assigned, and a resume used in the
    train split is removed from the pool available to the test split. This guarantees the
    fine-tuned model (trained on the train split) is evaluated only on job/resume pairs it
    never saw during training, avoiding train/test leakage.

    Returns (train_resumes, train_jobs, test_resumes, test_jobs).
    """
    occupation_catalog = load_esco_occupation_catalog()
    job_text = jobs['title'].fillna('') + ' ' + jobs['job_description'].fillna('').str.slice(0, 1200)
    mapped_jobs = pd.concat([jobs.reset_index(drop=True), map_texts_to_esco_occupations(job_text, occupation_catalog, config)], axis=1)
    mapped_resumes = pd.concat([resumes.reset_index(drop=True), map_texts_to_esco_occupations(resumes['resume_text'], occupation_catalog, config)], axis=1)
    mapped_jobs = mapped_jobs.loc[mapped_jobs['esco_mapping_score'] >= config.esco_mapping_min_score].copy()
    mapped_resumes = mapped_resumes.loc[mapped_resumes['esco_mapping_score'] >= config.esco_mapping_min_score].copy()

    shuffled_job_ids = mapped_jobs['id'].sample(frac=1.0, random_state=RANDOM_SEED).tolist()
    test_size = max(1, int(round(len(shuffled_job_ids) * config.test_fraction)))
    test_job_ids = set(shuffled_job_ids[:test_size])
    train_job_ids = set(shuffled_job_ids[test_size:])

    train_jobs, train_resumes = _build_job_resume_split(
        train_job_ids, mapped_jobs, mapped_resumes, resume_pool_seed_offset=1,
        max_jobs=config.max_jobs, max_resumes=config.max_resumes,
    )
    remaining_resumes = mapped_resumes[~mapped_resumes['id'].isin(set(train_resumes['id']))]
    test_jobs, test_resumes = _build_job_resume_split(
        test_job_ids, mapped_jobs, remaining_resumes, resume_pool_seed_offset=2,
        max_jobs=config.max_jobs, max_resumes=config.max_resumes,
    )

    if train_jobs.empty or test_jobs.empty:
        raise ValueError(
            'No ESCO-aligned job/resume pairs passed the mapping-confidence threshold in the '
            'train or test split; lower esco_mapping_min_score or increase max_jobs/max_resumes.'
        )
    return train_resumes, train_jobs, test_resumes, test_jobs


def export_esco_mapping_sample_for_annotation(
    mapped_jobs: pd.DataFrame, n: int, output_file: str, random_state: int = RANDOM_SEED
) -> pd.DataFrame:
    """Sample ESCO-mapped jobs for manual annotation of weak-supervision label correctness.

    Every ranking metric in this notebook depends on the ESCO occupation mapping being right.
    Fill in `human_label_correct` with 1 (the ESCO occupation genuinely matches the job title/
    description) or 0 (it does not), then call `evaluate_weak_label_agreement` to get a
    reportable precision estimate with a confidence interval for the thesis methodology section.
    """
    sample = mapped_jobs.sample(n=min(n, len(mapped_jobs)), random_state=random_state).copy()
    sample = sample[['id', 'title', 'esco_occupation_label', 'esco_mapping_score']]
    sample['human_label_correct'] = ''
    sample.to_csv(output_file, index=False)
    print(f'Wrote {len(sample)} rows to {output_file}. Fill in human_label_correct (1 or 0) for each row.')
    return sample


def evaluate_weak_label_agreement(annotated_csv_path: str) -> Dict[str, object]:
    """Compute the ESCO weak-supervision mapping's precision against human annotation.

    Uses a Wilson score interval (robust at small sample sizes) rather than a normal
    approximation, since annotation samples are typically only 30-100 rows.
    """
    annotated = pd.read_csv(annotated_csv_path)
    labels = pd.to_numeric(annotated['human_label_correct'], errors='coerce').dropna()
    if labels.empty:
        raise ValueError('No annotated rows found; fill in human_label_correct with 1 or 0 first.')
    labels = labels.astype(int)
    n = len(labels)
    correct = int(labels.sum())
    p_hat = correct / n
    z = 1.96
    denom = 1 + z ** 2 / n
    center = (p_hat + z ** 2 / (2 * n)) / denom
    half_width = (z * np.sqrt(p_hat * (1 - p_hat) / n + z ** 2 / (4 * n ** 2))) / denom
    return {
        'n_annotated': n,
        'precision': round(p_hat, 3),
        'ci_lower': round(float(max(0.0, center - half_width)), 3),
        'ci_upper': round(float(min(1.0, center + half_width)), 3),
    }


### Matching and evaluation functions

These functions create embeddings and similarity matrices and calculate Precision@K and NDCG@K.

In [ ]:
def build_tfidf_similarity_matrix(resumes: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:
    """TF-IDF cosine similarity matrix: (job count, resume count)."""
    corpus = resumes['resume_text'].tolist() + jobs['job_description'].tolist()
    vectors = TfidfVectorizer(stop_words='english').fit_transform(corpus)
    return cosine_similarity(vectors[len(resumes):], vectors[:len(resumes)])


def build_embedding_similarity_matrix(
    model: SentenceTransformer, resumes: pd.DataFrame, jobs: pd.DataFrame, batch_size: int
) -> np.ndarray:
    """Create a job–resume cosine-similarity matrix using SentenceTransformer."""
    resume_embeddings = model.encode(
        resumes['resume_text'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    job_embeddings = model.encode(
        jobs['job_description'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    return cosine_similarity(job_embeddings, resume_embeddings)


def calculate_ranking_metrics(
    similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    """Return Precision@K, NDCG@K, per-job arrays (for bootstrap/significance testing), and ranking details."""
    resume_ids = resumes['id'].tolist()
    hit_array, ndcg_array, details = [], [], []
    for row_index, job in jobs.iterrows():
        ranked_indices = np.argsort(similarity_matrix[row_index])[::-1]
        ranked_ids = [resume_ids[index] for index in ranked_indices]
        correct_id = job['correct_resume_id']
        rank = ranked_ids.index(correct_id) + 1 if correct_id in ranked_ids else None
        hit = rank is not None and rank <= top_k
        hit_array.append(int(hit))
        ndcg_array.append(1 / np.log2(rank + 1) if hit else 0.0)
        details.append({
            'job_title': job['title'], 'correct_resume': correct_id,
            'top1_predicted': ranked_ids[0],
            'top1_score': round(float(similarity_matrix[row_index, ranked_indices[0]]), 3),
            f'hit_at_{top_k}': hit,
        })
    hit_array, ndcg_array = np.array(hit_array, dtype=float), np.array(ndcg_array, dtype=float)
    return {
        f'Precision@{top_k}': round(float(hit_array.mean()), 3),
        f'NDCG@{top_k}': round(float(ndcg_array.mean()), 3),
        'hit_array': hit_array,
        'ndcg_array': ndcg_array,
        'details': pd.DataFrame(details),
    }


def evaluate_similarity_model(
    model_name: str, similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    metrics = calculate_ranking_metrics(similarity_matrix, jobs, resumes, top_k)
    return {'Model': model_name, **metrics}


def bootstrap_ci(values: np.ndarray, n_bootstrap: int, ci: float = 0.95, random_state: int = RANDOM_SEED) -> Tuple[float, float, float]:
    """Percentile bootstrap confidence interval for the mean of a per-job metric array."""
    rng = np.random.default_rng(random_state)
    n = len(values)
    boot_means = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        boot_means[i] = values[rng.integers(0, n, size=n)].mean()
    alpha = (1 - ci) / 2
    lower, upper = np.quantile(boot_means, [alpha, 1 - alpha])
    return float(values.mean()), float(lower), float(upper)


def paired_bootstrap_test(
    values_a: np.ndarray, values_b: np.ndarray, n_bootstrap: int, random_state: int = RANDOM_SEED
) -> Tuple[float, float]:
    """Two-sided paired bootstrap test for mean(values_a) != mean(values_b) on the same jobs.

    Returns (observed mean difference a-b, bootstrap p-value). Use this instead of eyeballing
    Precision@K/NDCG@K deltas between models: with only tens of jobs, a few points of difference
    can easily be noise, and a reviewer will ask for evidence the gap is not chance.
    """
    if len(values_a) != len(values_b):
        raise ValueError('values_a and values_b must be paired (same jobs, same order)')
    rng = np.random.default_rng(random_state)
    n = len(values_a)
    diffs = values_a - values_b
    observed_diff = float(diffs.mean())
    boot_diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        boot_diffs[i] = diffs[idx].mean()
    centered = boot_diffs - boot_diffs.mean()
    p_value = float(np.mean(np.abs(centered) >= np.abs(observed_diff)))
    return observed_diff, p_value


def summarize_model_comparison(results: List[Dict[str, object]], top_k: int, n_bootstrap: int) -> pd.DataFrame:
    """Build a reportable comparison table with bootstrap confidence intervals per model."""
    rows = []
    for result in results:
        precision_mean, precision_lo, precision_hi = bootstrap_ci(result['hit_array'], n_bootstrap)
        ndcg_mean, ndcg_lo, ndcg_hi = bootstrap_ci(result['ndcg_array'], n_bootstrap)
        rows.append({
            'Model': result['Model'],
            f'Precision@{top_k}': round(precision_mean, 3),
            f'Precision@{top_k} 95% CI': f'[{precision_lo:.3f}, {precision_hi:.3f}]',
            f'NDCG@{top_k}': round(ndcg_mean, 3),
            f'NDCG@{top_k} 95% CI': f'[{ndcg_lo:.3f}, {ndcg_hi:.3f}]',
        })
    return pd.DataFrame(rows)


def compare_models_significance(
    results: List[Dict[str, object]], reference_model: str, metric: str, n_bootstrap: int
) -> pd.DataFrame:
    """Paired bootstrap significance of every model against a reference model (e.g. the TF-IDF baseline)."""
    reference = next(result for result in results if result['Model'] == reference_model)
    rows = []
    for result in results:
        if result['Model'] == reference_model:
            continue
        diff, p_value = paired_bootstrap_test(result[metric], reference[metric], n_bootstrap)
        rows.append({
            'Model': result['Model'], 'vs': reference_model, f'{metric} diff': round(diff, 3),
            'p_value': round(p_value, 4), 'significant_at_0.05': p_value < 0.05,
        })
    return pd.DataFrame(rows)


### Fine-tuned model management

The model is loaded from Drive when available. Otherwise, it is trained and saved locally and to Drive.

In [ ]:
def mount_google_drive() -> bool:
    """Mount Google Drive only in Colab and return whether it is available."""
    try:
        from google.colab import drive
    except ImportError:
        print('Google Drive is unavailable outside Google Colab; skipping Drive.')
        return False
    drive.mount('/content/drive', force_remount=False)
    return True


def model_files_exist(model_path: str | Path) -> bool:
    """Check whether a directory contains the minimum SentenceTransformer model files."""
    path = Path(model_path)
    return path.is_dir() and (path / 'config_sentence_transformers.json').exists()


def create_training_examples(resumes: pd.DataFrame, jobs: pd.DataFrame) -> List[InputExample]:
    """Create ESCO-aligned positive pairs and different-occupation negative pairs."""
    resumes_by_occupation = resumes.groupby('esco_occupation_uri')['resume_text'].apply(list).to_dict()
    examples: List[InputExample] = []
    rng = random.Random(RANDOM_SEED)
    for _, job in jobs.iterrows():
        occupation_uri = job['esco_occupation_uri']
        positives = resumes_by_occupation.get(occupation_uri, [])
        if not positives:
            continue
        job_text = job['job_description'][:512]
        for resume_text in rng.sample(positives, k=min(2, len(positives))):
            examples.append(InputExample(texts=[job_text, resume_text[:512]], label=1.0))
        other_occupations = [key for key in resumes_by_occupation if key != occupation_uri]
        for negative_occupation in rng.sample(other_occupations, k=min(4, len(other_occupations))):
            negative_resume = rng.choice(resumes_by_occupation[negative_occupation])
            examples.append(InputExample(texts=[job_text, negative_resume[:512]], label=0.0))
    rng.shuffle(examples)
    if len(examples) < 2:
        raise ValueError('There are not enough training pairs for fine-tuning.')
    return examples


def train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> SentenceTransformer:
    """Fine-tune the base model and save the best model to `local_model_path`."""
    examples = create_training_examples(resumes, jobs)
    split_index = max(1, int(len(examples) * 0.9))
    train_examples, validation_examples = examples[:split_index], examples[split_index:]
    model = SentenceTransformer(config.base_model_name, device=config.device)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=config.batch_size)
    evaluator = None
    if validation_examples:
        evaluator = evaluation.EmbeddingSimilarityEvaluator(
            [item.texts[0] for item in validation_examples],
            [item.texts[1] for item in validation_examples],
            [item.label for item in validation_examples], name='validation'
        )
    model.fit(
        train_objectives=[(train_loader, losses.CosineSimilarityLoss(model))],
        evaluator=evaluator, epochs=config.epochs,
        warmup_steps=max(1, int(len(train_loader) * 0.1)),
        evaluation_steps=max(1, int(len(train_loader) * 0.5)),
        output_path=config.local_model_path, save_best_model=True, show_progress_bar=True,
    )
    return SentenceTransformer(config.local_model_path, device=config.device)


def load_or_train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[SentenceTransformer, str]:
    """Obtain the model in this order: Drive, local cache, then fine-tuning."""
    drive_is_available = mount_google_drive()
    if drive_is_available and model_files_exist(config.drive_model_path):
        print(f' Loaded fine-tuned model from Google Drive: {config.drive_model_path}')
        return SentenceTransformer(config.drive_model_path, device=config.device), 'Google Drive'
    if model_files_exist(config.local_model_path):
        print(f' Loaded fine-tuned model from local cache: {config.local_model_path}')
        return SentenceTransformer(config.local_model_path, device=config.device), 'local cache'

    print(' No saved fine-tuned model was found; training now.')
    model = train_finetuned_model(resumes, jobs, config)
    if drive_is_available:
        destination = Path(config.drive_model_path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(config.local_model_path, destination, dirs_exist_ok=True)
        print(f'Saved trained model to Google Drive: {destination}')
    return model, 'trained this run'


### Run All pipeline

The final cell below runs every step in dependency order. Adjust only `PipelineConfig` when needed.

In [ ]:
def evaluate_pretrained_models(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> List[Dict[str, object]]:
    """Collect results for TF-IDF and pretrained SBERT models on a held-out (test) split."""
    results = [evaluate_similarity_model(
        'TF-IDF (Baseline)', build_tfidf_similarity_matrix(resumes, jobs),
        jobs, resumes, config.top_k
    )]
    model_ids = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'BAAI/bge-base-en-v1.5']
    for model_id in model_ids:
        print(f'Evaluating: {model_id}')
        model = SentenceTransformer(model_id, device=config.device)
        similarity = build_embedding_similarity_matrix(model, resumes, jobs, config.batch_size)
        results.append(evaluate_similarity_model(model_id, similarity, jobs, resumes, config.top_k))
        del model, similarity
        gc.collect()
    return results


def run_job_matching_pipeline(config: PipelineConfig = CONFIG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run All entry point: load -> split -> fine-tune on train -> evaluate on held-out test -> save.

    Fine-tuning and evaluation use disjoint job/resume splits from `prepare_evaluation_data`, so
    the reported Precision@K/NDCG@K for the fine-tuned model reflect generalization rather than
    memorization of the evaluation pairs.
    """
    print('1/6 Loading resume and job-posting data')
    all_resumes = load_resume_dataset()
    all_jobs = load_job_postings()

    print('2/6 Preparing train/test-disjoint evaluation data')
    train_resumes, train_jobs, test_resumes, test_jobs = prepare_evaluation_data(all_resumes, all_jobs, config)
    print(f'   train: resumes={len(train_resumes)}, jobs={len(train_jobs)}')
    print(f'   test:  resumes={len(test_resumes)}, jobs={len(test_jobs)}')

    print('3/6 Obtaining the fine-tuned model (Google Drive first, else fine-tune on the train split only)')
    finetuned_model, model_source = load_or_train_finetuned_model(train_resumes, train_jobs, config)

    print('4/6 Evaluating baseline and pretrained models on the held-out test split')
    results = evaluate_pretrained_models(test_resumes, test_jobs, config)

    print('5/6 Evaluating the fine-tuned model on the held-out test split')
    finetuned_similarity = build_embedding_similarity_matrix(
        finetuned_model, test_resumes, test_jobs, config.batch_size
    )
    results.append(evaluate_similarity_model(
        f'SBERT Fine-tuned (JobLens; {model_source})', finetuned_similarity,
        test_jobs, test_resumes, config.top_k
    ))

    print('6/6 Computing bootstrap confidence intervals and significance tests, then saving results')
    comparison = summarize_model_comparison(results, config.top_k, config.n_bootstrap)
    significance = compare_models_significance(results, 'TF-IDF (Baseline)', 'ndcg_array', config.n_bootstrap)
    output_file = f'comparison_models_pool{len(test_resumes)}.csv'
    comparison.to_csv(output_file, index=False)
    significance.to_csv(f'significance_vs_baseline_pool{len(test_resumes)}.csv', index=False)
    print('\n' + comparison.to_string(index=False))
    print('\nSignificance vs. TF-IDF baseline (paired bootstrap on NDCG, held-out test jobs only):')
    print(significance.to_string(index=False))
    print(f'\n✅ Saved comparison results: {output_file}')
    return comparison, pd.DataFrame(results)[['Model', 'details']]


# This final Run All cell calls the functions in dependency order.
comparison_df, ranking_details_df = run_job_matching_pipeline()


## Part 2: Functional Skill Extraction Pipeline

The pipeline loads job postings once, extracts BERT NER and O*NET keyword skills in one pass, then produces hybrid results, summary metrics, and visualizations.

### Configuration and model setup

In [ ]:
import re
import time
from dataclasses import dataclass
from typing import Any

import matplotlib.pyplot as plt
from transformers import pipeline


@dataclass(frozen=True)
class SkillExtractionConfig:
    model_name: str = "GalalEwida/LLM-BERT-Model-Based-Skills-Extraction-from-jobdescription"
    sample_size: int = 100
    confidence_threshold: float = 0.7
    max_text_length: int = 500
    progress_interval: int = 10
    output_prefix: str = "skill_extraction"


SKILL_CONFIG = SkillExtractionConfig()


from pathlib import Path


def load_esco_skills() -> list[str]:
    """Load normalized ESCO preferred and alternative skill labels with pandas."""
    candidate_paths = (
        Path("/colab/data/skills_en.csv"),
        Path("colab/data/skills_en.csv"),
        Path("data/skills_en.csv"),
    )
    esco_path = next((path for path in candidate_paths if path.exists()), None)
    if esco_path is None:
        raise FileNotFoundError("ESCO skills_en.csv not found in /colab/data, colab/data, or data")

    # Include ESCO's preferred and alternative labels so, for example,
    # "Python" matches the canonical "python (computer programming)" skill.
    # Exact token/phrase boundaries are enforced during extraction below.
    esco_skills_df = pd.read_csv(esco_path, usecols=["preferredLabel", "altLabels", "hiddenLabels"])
    labels = pd.concat([
        esco_skills_df["preferredLabel"],
        esco_skills_df["altLabels"].fillna("").str.split("\n").explode(),
        esco_skills_df["hiddenLabels"].fillna("").str.split("\n").explode(),
    ], ignore_index=True)
    return (labels.dropna().astype(str).str.strip().str.lower()
            .loc[lambda values: values.str.len() >= 3].drop_duplicates().tolist())


ESCO_SKILLS = load_esco_skills()
# Kept for compatibility with the existing extraction and evaluation code.
ESCO_SKILLS_SET = sorted(ESCO_SKILLS, key=len, reverse=True)
# Match complete tokens/phrases only: e.g. "ski" must not match "skills".
ESCO_SKILL_PATTERNS = [
    (skill, re.compile(rf"(?<![A-Za-z0-9]){re.escape(skill)}(?![A-Za-z0-9])"))
    for skill in ESCO_SKILLS_SET
]
print(f"Loaded {len(ESCO_SKILLS):,} ESCO skill labels")

In [ ]:
def load_skill_extraction_jobs(config: SkillExtractionConfig) -> pd.DataFrame:
    """Load and reproducibly sample the shared LinkedIn job-posting dataset."""
    jobs = load_job_postings()
    return jobs.sample(n=min(config.sample_size, len(jobs)), random_state=RANDOM_SEED).reset_index(drop=True)


def load_skill_ner_model(config: SkillExtractionConfig) -> Any:
    """Load the JobBERT NER pipeline on CPU; set `device=0` here to use a GPU."""
    return pipeline(
        "token-classification",
        model=config.model_name,
        aggregation_strategy="simple",
        device=-1,
    )


def extract_bert_skills(text: str, ner_model: Any, config: SkillExtractionConfig) -> list[str]:
    """Extract unique, high-confidence skills from one job description using BERT NER."""
    entities = ner_model(str(text)[:config.max_text_length])
    return list(dict.fromkeys(
        entity["word"].strip().lower()
        for entity in entities
        if entity["score"] >= config.confidence_threshold and len(entity["word"].strip()) >= 2
    ))


def extract_onet_skills(text: str) -> list[str]:
    """Extract non-overlapping ESCO dictionary skills from one job description."""
    normalized_text = str(text).lower()
    matched, used_positions = [], set()
    for skill, pattern in ESCO_SKILL_PATTERNS:
        match = pattern.search(normalized_text)
        if match is None:
            continue
        start, end = match.span()
        positions = set(range(start, end))
        if positions & used_positions:
            continue
        matched.append(skill)
        used_positions.update(positions)
    return matched


def combine_skills(bert_skills: list[str], onet_skills: list[str], mode: str = "intersection") -> list[str]:
    """Combine BERT and ESCO results using intersection or union mode."""
    if mode == "union":
        return sorted(set(bert_skills) | set(onet_skills))
    if mode != "intersection":
        raise ValueError("mode must be either 'intersection' or 'union'")
    return sorted({
        onet_skill for onet_skill in onet_skills
        if any(bert_skill in onet_skill or onet_skill in bert_skill for bert_skill in bert_skills)
    })


def extract_skill_profile(
    text: str, ner_model: Any, config: SkillExtractionConfig, hybrid_mode: str = "union"
) -> dict[str, list[str]]:
    """Extract BERT, ESCO, and hybrid skills from one job description or resume.
    This is the shared Part 2 extraction interface used by both batch evaluation
    and the single resume–job analysis in Part 3.
    """
    try:
        bert_skills = extract_bert_skills(text, ner_model, config)
    except Exception as error:
        print(f"BERT skill extraction skipped: {error}")
        bert_skills = []
    onet_skills = extract_onet_skills(text)
    return {
        "bert_skills": bert_skills,
        "esco_skills": onet_skills,
        "hybrid_skills": combine_skills(bert_skills, onet_skills, mode=hybrid_mode),
    }

### Extraction, evaluation, and visualization functions

In [ ]:
def extract_skills_for_jobs(
    jobs: pd.DataFrame, ner_model: Any, config: SkillExtractionConfig, hybrid_mode: str = "intersection"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run BERT NER and ESCO extraction once per job and create all result tables."""
    records = []
    for index, (_, job) in enumerate(jobs.iterrows(), start=1):
        skill_profile = extract_skill_profile(job["job_description"], ner_model, config, hybrid_mode)
        records.append({
            "job_id": job["id"], "job_title": job["title"],
            "bert_skills": skill_profile["bert_skills"],
            "esco_skills": skill_profile["esco_skills"],
            "hybrid_skills": skill_profile["hybrid_skills"],
            "n_bert": len(skill_profile["bert_skills"]),
            "n_esco": len(skill_profile["esco_skills"]),
            "n_hybrid": len(skill_profile["hybrid_skills"]),
        })
        if index % config.progress_interval == 0 or index == len(jobs):
            print(f"Progress: {index}/{len(jobs)} postings")

    extracted = pd.DataFrame(records)
    bert_results = extracted[["job_id", "job_title", "bert_skills", "n_bert"]].rename(
        columns={"bert_skills": "all_skills", "n_bert": "n_skills"}
    )
    esco_results = extracted[["job_id", "job_title", "esco_skills", "n_esco"]].rename(
        columns={"esco_skills": "all_skills", "n_esco": "n_skills"}
    )
    return bert_results, esco_results, extracted


def build_skill_comparison(
    bert_results: pd.DataFrame, esco_results: pd.DataFrame, hybrid_results: pd.DataFrame
) -> pd.DataFrame:
    """Calculate coverage and skill-count statistics for the three extraction methods.

    These are descriptive statistics only (how many skills each method finds), not accuracy.
    Use `export_skill_annotation_sample` + `evaluate_skill_extraction_against_gold` below to get
    precision/recall/F1 against human-annotated gold skills before claiming one method is "better."
    """
    method_counts = {
        "BERT NER": bert_results["n_skills"],
        "ESCO Keyword Matching": esco_results["n_skills"],
        "Hybrid": hybrid_results["n_hybrid"],
    }
    return pd.DataFrame({
        "Method": method_counts.keys(),
        "Coverage (%)": [round((counts > 0).mean() * 100, 1) for counts in method_counts.values()],
        "Avg Skills/Job": [round(counts.mean(), 1) for counts in method_counts.values()],
        "Max Skills/Job": [int(counts.max()) for counts in method_counts.values()],
        "Median Skills/Job": [round(counts.median(), 1) for counts in method_counts.values()],
    })


def export_skill_annotation_sample(extracted: pd.DataFrame, n: int, output_file: str, random_state: int = RANDOM_SEED) -> pd.DataFrame:
    """Sample job postings for gold-standard skill annotation, independent of any extractor's output.

    List every skill the job actually requires or prefers in `gold_skills` (comma-separated),
    without looking at the extractor outputs first, so recall is measured honestly.
    """
    sample = extracted[["job_id", "job_title"]].sample(n=min(n, len(extracted)), random_state=random_state).copy()
    sample["gold_skills"] = ""
    sample.to_csv(output_file, index=False)
    print(f"Wrote {len(sample)} rows to {output_file}. Fill in gold_skills (comma-separated) for each row.")
    return sample


def _parse_gold_skill_list(cell: object) -> set[str]:
    if pd.isna(cell) or not str(cell).strip():
        return set()
    return {item.strip().lower() for item in str(cell).split(",") if item.strip()}


def evaluate_skill_extraction_against_gold(extracted: pd.DataFrame, annotated_csv_path: str) -> pd.DataFrame:
    """Compute precision/recall/F1 for BERT NER, ESCO keyword, and hybrid extraction against gold skills."""
    gold = pd.read_csv(annotated_csv_path)
    gold["gold_set"] = gold["gold_skills"].map(_parse_gold_skill_list)
    gold = gold[gold["gold_set"].map(len) > 0]
    if gold.empty:
        raise ValueError("No annotated rows found; fill in gold_skills for at least one job.")

    merged = gold.merge(extracted, on="job_id", how="left", suffixes=("", "_extracted"))
    method_columns = {"BERT NER": "bert_skills", "ESCO Keyword Matching": "esco_skills", "Hybrid": "hybrid_skills"}
    rows = []
    for method_name, column in method_columns.items():
        precisions, recalls = [], []
        for _, row in merged.iterrows():
            predicted = {str(skill).strip().lower() for skill in (row[column] or [])}
            gold_set = row["gold_set"]
            true_positives = len(predicted & gold_set)
            precisions.append(true_positives / len(predicted) if predicted else 0.0)
            recalls.append(true_positives / len(gold_set) if gold_set else 0.0)
        precision, recall = sum(precisions) / len(precisions), sum(recalls) / len(recalls)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "Method": method_name, "Precision": round(precision, 3),
            "Recall": round(recall, 3), "F1": round(f1, 3), "n_annotated": len(merged),
        })
    return pd.DataFrame(rows)


def plot_skill_comparison(
    comparison: pd.DataFrame, bert_results: pd.DataFrame, esco_results: pd.DataFrame, hybrid_results: pd.DataFrame, output_file: str
) -> None:
    """Create and save coverage, average-count, and distribution comparison charts."""
    colors = ["#2E5FA3", "#27AE60", "#E67E22"]
    labels = comparison["Method"].tolist()
    fig, axes = plt.subplots(1, 3, figsize=(17, 6))
    fig.suptitle("JobLens AI — Skill Extraction Method Comparison", fontsize=14, fontweight="bold")

    for axis, column, title, ylabel in [
        (axes[0], "Coverage (%)", "Coverage", "Coverage (%)"),
        (axes[1], "Avg Skills/Job", "Average Skills per Job", "Skills"),
    ]:
        bars = axis.bar(labels, comparison[column], color=colors)
        axis.set_title(title)
        axis.set_ylabel(ylabel)
        axis.tick_params(axis="x", rotation=20)
        for bar, value in zip(bars, comparison[column]):
            axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value}", ha="center", va="bottom")

    axes[2].boxplot([bert_results["n_skills"], esco_results["n_skills"], hybrid_results["n_hybrid"]], labels=labels)
    axes[2].set_title("Skill-count Distribution")
    axes[2].set_ylabel("Skills")
    axes[2].tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.show()


def run_skill_extraction_pipeline(
    config: SkillExtractionConfig = SKILL_CONFIG, hybrid_mode: str = "intersection", annotation_sample_size: int = 30
) -> dict[str, pd.DataFrame]:
    """Run the complete functional skill-extraction workflow and save its artifacts.

    Also exports a gold-annotation sample CSV so the descriptive comparison above can be
    followed up with real precision/recall/F1 numbers via `evaluate_skill_extraction_against_gold`.
    """
    print("1/4 Loading job postings")
    jobs = load_skill_extraction_jobs(config)
    print(f"   Sampled {len(jobs)} postings")

    print("2/4 Loading BERT NER model")
    ner_model = load_skill_ner_model(config)

    print("3/4 Extracting BERT NER, ESCO, and hybrid skills")
    bert_results, esco_results, hybrid_results = extract_skills_for_jobs(jobs, ner_model, config, hybrid_mode)

    print("4/4 Building comparison, exporting annotation sample, and saving artifacts")
    comparison = build_skill_comparison(bert_results, esco_results, hybrid_results)
    bert_results.to_csv(f"{config.output_prefix}_bert.csv", index=False)
    esco_results.to_csv(f"{config.output_prefix}_esco.csv", index=False)
    hybrid_results.to_csv(f"{config.output_prefix}_hybrid.csv", index=False)
    comparison.to_csv(f"{config.output_prefix}_comparison.csv", index=False)
    export_skill_annotation_sample(hybrid_results, annotation_sample_size, f"{config.output_prefix}_gold_annotation_template.csv")
    plot_skill_comparison(
        comparison, bert_results, esco_results, hybrid_results, f"{config.output_prefix}_comparison.png"
    )
    print(comparison.to_string(index=False))
    print(
        f"\nTo report precision/recall/F1: fill in gold_skills in "
        f"{config.output_prefix}_gold_annotation_template.csv, then call "
        f"evaluate_skill_extraction_against_gold(hybrid_results, '{config.output_prefix}_gold_annotation_template.csv')."
    )
    return {
        "jobs": jobs, "bert_results": bert_results, "esco_results": esco_results,
        "hybrid_results": hybrid_results, "comparison": comparison,
    }


### Run All

In [ ]:
# Run the complete Part 2 workflow.
skill_extraction_outputs = run_skill_extraction_pipeline()
skill_extraction_outputs["comparison"]

## Part 3: Single Resume–Job Analysis

Use this section with one user-provided job description and one user-provided resume. It produces an explainable JobLens Match Score and a skill-gap analysis. The score is a job-fit indicator, not a hiring prediction.

In [ ]:
@dataclass(frozen=True)
class ApplicationAnalysisConfig:
    similarity_model_name: str = "all-MiniLM-L6-v2"
    semantic_weight: float = 0.50
    skill_weight: float = 0.35
    # requirement_weight is a future-work extension point: no requirement_match_score
    # extractor exists yet, so calculate_match_score always renormalizes over the two
    # weights above. Implement a requirement extractor before reporting this weight as used.
    requirement_weight: float = 0.15


APPLICATION_CONFIG = ApplicationAnalysisConfig()


from pathlib import Path


def _normalize_esco_label(label: str) -> str:
    return " ".join(str(label).lower().replace("##", "").split())


def _esco_data_path(filename: str) -> Path:
    candidate_paths = (Path("/colab/data"), Path("colab/data"), Path("data"))
    path = next((base / filename for base in candidate_paths if (base / filename).exists()), None)
    if path is None:
        raise FileNotFoundError(f"ESCO data file not found: {filename}")
    return path


def load_esco_skill_aliases() -> dict[str, str]:
    """Build unambiguous skill aliases from ESCO labels, relations, and hierarchy."""
    skills = pd.read_csv(
        _esco_data_path("skills_en.csv"),
        usecols=["conceptUri", "preferredLabel", "altLabels", "hiddenLabels"],
    )
    relations = pd.read_csv(
        _esco_data_path("occupationSkillRelations_en.csv"),
        usecols=["skillUri"],
    )
    hierarchy = pd.read_csv(_esco_data_path("skillsHierarchy_en.csv"))

    # Only retain skills linked to at least one ESCO occupation.
    relation_counts = relations["skillUri"].value_counts()
    skills = skills.loc[skills["conceptUri"].isin(relation_counts.index)].copy()
    skills["canonical"] = skills["preferredLabel"].map(_normalize_esco_label)

    # Do not map a taxonomy/category name to a more specific child skill.
    hierarchy_label_columns = [
        f"Level {level} preferred term" for level in range(4)
        if f"Level {level} preferred term" in hierarchy.columns
    ]
    hierarchy_terms = set(
        hierarchy[hierarchy_label_columns].stack().map(_normalize_esco_label)
    )

    alias_rows = []
    for column in ("altLabels", "hiddenLabels"):
        alias_rows.append(skills[["canonical", column]].assign(
            alias=lambda frame: frame[column].fillna("").str.split("\n")
        ).explode("alias")[["canonical", "alias"]])
    aliases = pd.concat(alias_rows, ignore_index=True)
    aliases["alias"] = aliases["alias"].map(_normalize_esco_label)
    aliases = aliases.loc[
        (aliases["alias"].str.len() >= 2)
        & (aliases["alias"] != aliases["canonical"])
        & ~aliases["alias"].isin(hierarchy_terms)
    ]

    # Retain aliases only when ESCO maps them to exactly one canonical skill.
    unique_aliases = aliases.groupby("alias")["canonical"].agg(lambda values: set(values))
    return {alias: next(iter(canonicals)) for alias, canonicals in unique_aliases.items() if len(canonicals) == 1}


SKILL_ALIASES = load_esco_skill_aliases()
print(f"Loaded {len(SKILL_ALIASES):,} unambiguous ESCO skill aliases")


def normalize_skill(skill: str) -> str:
    """Normalize a skill label to support reliable matching across common variants."""
    normalized = " ".join(str(skill).lower().replace("##", "").split())
    return SKILL_ALIASES.get(normalized, normalized)


def normalize_skills(skills: list[str]) -> list[str]:
    """Normalize, remove blank values, and preserve the first occurrence of each skill."""
    return list(dict.fromkeys(
        normalized for skill in skills
        if (normalized := normalize_skill(skill))
    ))

In [ ]:
def extract_document_skills(
    text: str, ner_model: Any, skill_config: SkillExtractionConfig
) -> dict[str, list[str]]:
    """Extract normalized BERT, O*NET, and hybrid skill lists from any text document."""
    skill_profile = extract_skill_profile(text, ner_model, skill_config, hybrid_mode="union")
    return {
        method: normalize_skills(skills)
        for method, skills in skill_profile.items()
    }


def extract_job_skills(job_description: str, ner_model: Any, skill_config: SkillExtractionConfig) -> list[str]:
    """Extract the normalized skill set required or preferred by a job description."""
    return extract_document_skills(job_description, ner_model, skill_config)["hybrid_skills"]


def extract_resume_skills(resume_text: str, ner_model: Any, skill_config: SkillExtractionConfig) -> list[str]:
    """Extract the normalized skill set evidenced in a resume."""
    return extract_document_skills(resume_text, ner_model, skill_config)["hybrid_skills"]


def compare_skills(job_skills: list[str], resume_skills: list[str]) -> dict[str, list[str]]:
    """Return matched, missing, and additional skills in a stable alphabetical order."""
    job_set, resume_set = set(job_skills), set(resume_skills)
    return {
        "matched_skills": sorted(job_set & resume_set),
        "missing_skills": sorted(job_set - resume_set),
        "additional_resume_skills": sorted(resume_set - job_set),
    }


def calculate_semantic_similarity_score(
    job_description: str, resume_text: str, similarity_model: SentenceTransformer
) -> float:
    """Calculate a 0–100 cosine-similarity score for one job and one resume."""
    embeddings = similarity_model.encode([job_description, resume_text], show_progress_bar=False)
    cosine_score = float(cosine_similarity([embeddings[0]], [embeddings[1]])[0, 0])
    return round(max(0.0, min(1.0, cosine_score)) * 100, 1)


def calculate_skill_match_score(job_skills: list[str], resume_skills: list[str]) -> float | None:
    """Calculate job-skill coverage as a percentage; return None when no job skills are found."""
    if not job_skills:
        return None
    return round(len(set(job_skills) & set(resume_skills)) / len(set(job_skills)) * 100, 1)


def calculate_match_score(
    semantic_similarity_score: float, skill_match_score: float | None,
    requirement_match_score: float | None = None, config: ApplicationAnalysisConfig = APPLICATION_CONFIG
) -> dict[str, float | None]:
    """Combine available score components and renormalize weights for unavailable components."""
    components = {
        "semantic_similarity_score": (semantic_similarity_score, config.semantic_weight),
        "skill_match_score": (skill_match_score, config.skill_weight),
        "requirement_match_score": (requirement_match_score, config.requirement_weight),
    }
    available_weight = sum(weight for score, weight in components.values() if score is not None)
    if not available_weight:
        raise ValueError("At least one score component is required.")
    match_score = sum(score * weight for score, weight in components.values() if score is not None) / available_weight
    return {"match_score": round(match_score, 1), **{name: score for name, (score, _) in components.items()}}


def build_recommendations(skill_comparison: dict[str, list[str]], match_score: float, target_score: float) -> list[str]:
    """Create factual, prioritized recommendations without suggesting invented experience."""
    recommendations = []
    missing = skill_comparison["missing_skills"]
    if missing:
        recommendations.append(
            "Review these missing job skills and add them only when supported by real experience: " + ", ".join(missing)
        )
    if skill_comparison["matched_skills"]:
        recommendations.append(
            "Move relevant evidence for these matched skills closer to the professional summary and experience bullets: "
            + ", ".join(skill_comparison["matched_skills"])
        )
    if match_score < target_score:
        recommendations.append(
            f"The current score is below the {target_score:.0f}% target. Address real skill or experience gaps rather than adding unsupported claims."
        )
    if not recommendations:
        recommendations.append("No skill gaps were detected by the current extractors; review the semantic alignment of the resume language.")
    return recommendations

In [ ]:
def analyze_application(
    job_description: str, resume_text: str, target_score: float = 90.0,
    skill_config: SkillExtractionConfig = SKILL_CONFIG,
    analysis_config: ApplicationAnalysisConfig = APPLICATION_CONFIG,
    ner_model: Any | None = None, similarity_model: SentenceTransformer | None = None,
) -> dict[str, object]:
    """Analyze one user-provided job description and resume with explainable outputs."""
    if not job_description or not str(job_description).strip():
        raise ValueError("job_description must not be empty.")
    if not resume_text or not str(resume_text).strip():
        raise ValueError("resume_text must not be empty.")

    ner_model = ner_model or load_skill_ner_model(skill_config)
    similarity_model = similarity_model or SentenceTransformer(analysis_config.similarity_model_name)
    job_skills = extract_job_skills(job_description, ner_model, skill_config)
    resume_skills = extract_resume_skills(resume_text, ner_model, skill_config)
    skill_comparison = compare_skills(job_skills, resume_skills)
    semantic_score = calculate_semantic_similarity_score(job_description, resume_text, similarity_model)
    skill_score = calculate_skill_match_score(job_skills, resume_skills)
    score_breakdown = calculate_match_score(semantic_score, skill_score, config=analysis_config)

    return {
        **score_breakdown,
        "target_score": target_score,
        "job_skills": job_skills,
        "resume_skills": resume_skills,
        **skill_comparison,
        "recommendations": build_recommendations(skill_comparison, score_breakdown["match_score"], target_score),
    }


def display_application_analysis(analysis: dict[str, object]) -> None:
    """Print a compact, readable summary of a single application analysis."""
    print(f"JobLens Match Score: {analysis['match_score']}% (target: {analysis['target_score']}%)")
    print(f"Semantic similarity: {analysis['semantic_similarity_score']}%")
    print(f"Skill match: {analysis['skill_match_score']}%" if analysis["skill_match_score"] is not None else "Skill match: unavailable")
    print("\nMatched skills:", ", ".join(analysis["matched_skills"]) or "None detected")
    print("Missing skills:", ", ".join(analysis["missing_skills"]) or "None detected")
    print("\nRecommendations:")
    for recommendation in analysis["recommendations"]:
        print(f"- {recommendation}")

### Analyze user-provided text

In [ ]:
# Paste the job description and resume below, then uncomment the final two lines.
JOB_DESCRIPTION = """
As a Software Development Engineer for Amazon SES, you'll work on backend and customer-facing systems, contributing to the development and improvement of our email services. You'll focus on building reliable and scalable solutions, collaborating with other engineers to solve complex technical challenges, and ensuring our services are highly available and scalable to meet the needs of our global customer base.

As a member of the SES engineering team, your job will allow you to work with customers and other technology leaders at Amazon to translate strategic business needs into features and projects, with an opportunity to participate in strategic planning, contributing to the overall direction of the service.

Leadership and Culture-Building:
In Amazon everyone is a leader. You will contribute to fostering a culture of creativity and openness. Your work will encourage innovation and growth, and you will play a role in mentoring and supporting your colleagues. Knowledge sharing and professional development are important, and you'll be part of a team that values these elements.

On-Call Responsibility:
This position involves on-call responsibilities. We don’t like being paged in the middle of the night or on the weekend, so we work to ensure that our systems are fault-tolerant. When we do get paged, we work together to resolve the root cause so that we aren’t paged for the same issue twice.

AWS Utility Computing (UC) provides product innovations — from foundational services such as Amazon’s Simple Storage Service (S3) and Amazon Elastic Compute Cloud (EC2), to consistently released new product innovations that continue to set AWS’s services and features apart in the industry. As a member of the UC organization, you’ll support the development and management of Compute, Database, Storage, Internet of Things (Iot), Platform, and Productivity Apps services in AWS, including support for customers who require specialized security solutions for their cloud services.

"""

RESUME_TEXT = """
• Developed web applications integrating Oracle databases with backend business systems
• Designed data processing workflows for enterprise-scale manufacturing and operational datasets
• Built dashboards supporting production monitoring, sales analysis, and operational reporting
• Implemented ERP-MES integration interfaces to automate business processes and system communication
• Automated Production-Sales-Inventory (PSI) workflows to reduce repetitive manual operations
• Optimized Oracle SQL queries and data retrieval processes for enterprise reporting systems
• Worked closely with business analysts and developers to deliver production-ready software solutions
• Developed search services using Java, Spring MVC, and RESTful APIs
• Implemented filtering, sorting, keyword highlighting, and search optimization features
• Built internal workflow automation tools to improve development efficiency
• Developed keyword extraction and analytics systems supporting marketing and customer insight analysis
• Designed KPI dashboards and reporting tools using enterprise datasets
• Automated deployment and development environments using Shell scripting
• Participated in debugging, production support, feature enhancement, and software maintenance
"""

# application_analysis = analyze_application(JOB_DESCRIPTION, RESUME_TEXT)
# display_application_analysis(application_analysis)

## Part 4: Truthful Resume Tailoring

This section is provider-agnostic: supply a function that calls your chosen LLM. JobLens builds a strict prompt and validates the resulting draft before it is displayed. Never show an unvalidated generated resume as a final version.

In [ ]:
import re
from collections.abc import Callable


def build_resume_tailoring_prompt(
    job_description: str, original_resume: str, application_analysis: dict[str, object]
) -> str:
    """Build a fact-preserving prompt for a provider-specific resume rewriter."""
    matched_skills = ", ".join(application_analysis["matched_skills"]) or "None detected"
    missing_skills = ", ".join(application_analysis["missing_skills"]) or "None detected"
    return f"""You are revising a resume for a specific job.

Non-negotiable truthfulness rules:
- Use only facts, skills, employers, job titles, dates, credentials, projects, and achievements found in the original resume.
- Do not invent experience, qualifications, metrics, certifications, or proficiency.
- Do not add missing skills unless the original resume explicitly supports them.
- If a required skill is absent, omit it from the revised resume.

Verified matched skills that must be preserved: {matched_skills}
Missing skills to avoid claiming: {missing_skills}

Skill-preservation requirements:
- Preserve clear, ATS-readable evidence for every verified matched skill listed above.
- Do not delete a verified matched skill, replace it with vague wording, or omit it while shortening the resume.
- Retain the original resume's wording for a skill whenever possible; use an equivalent ESCO alias only when it is supported by the original resume.
- Place the retained skill in the Skills section or in the relevant experience/project bullet, without inventing new experience.
- Do not introduce, infer, recommend, or mention any skill that is not explicitly present in the original resume, even if it appears in the job description.
- Do not include missing-job-skill names in a summary, skills list, headline, objective, or experience bullet.
- If uncertain whether a skill is supported by the original resume, omit it rather than adding it.


Job description:
{job_description}

Original resume:
{original_resume}

Return only the revised resume in a clear, ATS-friendly format.
"""


def generate_improved_resume(
    job_description: str, original_resume: str, application_analysis: dict[str, object],
    rewrite_resume: Callable[[str], str],
) -> str:
    """Generate a tailored resume through a caller-provided LLM function."""
    prompt = build_resume_tailoring_prompt(job_description, original_resume, application_analysis)
    revised_resume = rewrite_resume(prompt)
    if not revised_resume or not revised_resume.strip():
        raise ValueError("The resume rewriter returned an empty result.")
    return revised_resume.strip()


def extract_numeric_claims(text: str) -> set[str]:
    """Extract years, percentages, currency values, and count-based claims for review."""
    patterns = [r"\b(?:19|20)\d{2}\b", r"\b\d+(?:\.\d+)?%", r"[$€£]\s?\d[\d,]*(?:\.\d+)?", r"\b\d+\+?\s+(?:years?|months?|people|users|customers|projects?)\b"]
    return {match.lower() for pattern in patterns for match in re.findall(pattern, text, flags=re.IGNORECASE)}


def validate_resume_claims(
    original_resume: str, revised_resume: str, ner_model: Any, skill_config: SkillExtractionConfig
) -> dict[str, object]:
    """Flag new, exact ESCO keyword skills and quantitative claims for confirmation."""
    # BERT can label ordinary prose differently after a rewrite. Claim validation
    # therefore compares only strict, boundary-aware ESCO keyword matches.
    original_skills = set(normalize_skills(extract_onet_skills(original_resume)))
    revised_skills = set(normalize_skills(extract_onet_skills(revised_resume)))
    new_skills = sorted(revised_skills - original_skills)
    new_numeric_claims = sorted(extract_numeric_claims(revised_resume) - extract_numeric_claims(original_resume))
    return {
        "is_safe_to_show": not new_skills and not new_numeric_claims,
        "new_skills_requiring_confirmation": new_skills,
        "new_numeric_claims_requiring_confirmation": new_numeric_claims,
    }

In [ ]:
def improve_and_reanalyze_resume(
    job_description: str, original_resume: str, rewrite_resume: Callable[[str], str],
    target_score: float = 90.0, skill_config: SkillExtractionConfig = SKILL_CONFIG,
    analysis_config: ApplicationAnalysisConfig = APPLICATION_CONFIG,
    ner_model: Any | None = None, similarity_model: SentenceTransformer | None = None,
) -> dict[str, object]:
    """Generate, validate, and score a tailored-resume draft with review warnings."""
    ner_model = ner_model or load_skill_ner_model(skill_config)
    similarity_model = similarity_model or SentenceTransformer(analysis_config.similarity_model_name)
    before = analyze_application(
        job_description, original_resume, target_score, skill_config, analysis_config, ner_model, similarity_model
    )
    tailored_resume = generate_improved_resume(job_description, original_resume, before, rewrite_resume)
    validation = validate_resume_claims(original_resume, tailored_resume, ner_model, skill_config)
    after = analyze_application(
        job_description, tailored_resume, target_score, skill_config, analysis_config, ner_model, similarity_model
    )
    return {
        "before": before, "tailored_resume": tailored_resume, "validation": validation, "after": after,
        "requires_claim_review": not validation["is_safe_to_show"],
        "target_reached": after["match_score"] >= target_score,
        "message": (
            "Tailored resume scored, but flagged claims require review before use."
            if not validation["is_safe_to_show"]
            else "Tailored resume validated and re-evaluated."
        ),
    }


def display_before_after_comparison(result: dict[str, object]) -> None:
    """Display score changes and any validation action required after resume tailoring."""
    print(f"Before: {result['before']['match_score']}%")
    if result["after"] is None:
        print("After: not calculated because the draft needs claim review.")
        print("New skills requiring confirmation:", result["validation"]["new_skills_requiring_confirmation"])
        print("New numeric claims requiring confirmation:", result["validation"]["new_numeric_claims_requiring_confirmation"])
        return
    print(f"After: {result['after']['match_score']}%")
    print(f"Target reached: {result['target_reached']}")
    if result.get("requires_claim_review", False):
        print("Warning: review flagged claims before using this draft.")
        print("New skills requiring confirmation:", result["validation"]["new_skills_requiring_confirmation"])
        print("New numeric claims requiring confirmation:", result["validation"]["new_numeric_claims_requiring_confirmation"])


def download_tailored_resume(result: dict[str, object], filename: str | None = None) -> Path:
    """Save the tailored resume and trigger a browser download in Google Colab."""
    review_required = result.get("requires_claim_review", result["after"] is None)
    filename = filename or (
        "tailored_resume_review_required.txt" if review_required else "tailored_resume_validated.txt"
    )
    output_path = Path(filename)
    output_path.write_text(str(result["tailored_resume"]), encoding="utf-8")
    if review_required:
        print(f"Saved review-required draft: {output_path}")
    else:
        print(f"Saved validated tailored resume: {output_path}")

    try:
        from google.colab import files
    except ImportError:
        print("Download is available in Google Colab; use the saved file path in another environment.")
    else:
        files.download(str(output_path))
    return output_path


@dataclass(frozen=True)
class OpenRouterConfig:
    # "openrouter/free" auto-routes to whichever free model is available and can change
    # over time, which breaks reproducibility. Pin an exact model ID (e.g.
    # "meta-llama/llama-3.1-8b-instruct") before running any experiment reported in the thesis,
    # and record the exact model string alongside the results.
    model: str = "openrouter/free"
    base_url: str = "https://openrouter.ai/api/v1"
    max_tokens: int = 1_500
    temperature: float = 0.2
    empty_response_retries: int = 2
    app_title: str = "JobLens AI"
    dotenv_path: str = ".env"


OPENROUTER_CONFIG = OpenRouterConfig()
if OPENROUTER_CONFIG.model == "openrouter/free":
    print(
        "Warning: OPENROUTER_CONFIG.model is the 'openrouter/free' auto-routing alias, which is "
        "not reproducible (the underlying model can change over time). Pin an exact model ID "
        "before running any experiment you plan to report in the thesis."
    )


def load_openrouter_api_key(config: OpenRouterConfig = OPENROUTER_CONFIG) -> str:
    """Load the OpenRouter key from a local .env file without exposing it in the notebook."""
    import os
    try:
        from dotenv import load_dotenv
    except ImportError as error:
        raise ImportError("Install python-dotenv first: pip install python-dotenv") from error

    load_dotenv(config.dotenv_path)
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY was not found. Copy .env.example to .env and add your OpenRouter API key."
        )
    return api_key


def rewrite_with_your_llm(prompt: str, config: OpenRouterConfig = OPENROUTER_CONFIG) -> str:
    """Generate a tailored resume through OpenRouter using the configured API key and model."""
    try:
        from openai import OpenAI
    except ImportError as error:
        raise ImportError("Install the OpenAI-compatible client first: pip install openai") from error

    client = OpenAI(
        base_url=config.base_url,
        api_key=load_openrouter_api_key(config),
        default_headers={"X-OpenRouter-Title": config.app_title},
    )
    messages = [
        {
            "role": "system",
            "content": (
                "You revise resumes while preserving factual accuracy. "
                "Follow every truthfulness rule in the user prompt and return only the revised resume."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    last_finish_reason = "no choices returned"
    for attempt in range(config.empty_response_retries + 1):
        completion = client.chat.completions.create(
            model=config.model, messages=messages, temperature=config.temperature, max_tokens=config.max_tokens,
        )
        choice = completion.choices[0] if completion.choices else None
        revised_resume = choice.message.content if choice else None
        if revised_resume and revised_resume.strip():
            return revised_resume.strip()
        last_finish_reason = getattr(choice, "finish_reason", last_finish_reason)
        if attempt < config.empty_response_retries:
            time.sleep(1)

    raise RuntimeError(
        f"OpenRouter returned no resume text after {config.empty_response_retries + 1} attempts "
        f"(model={config.model!r}, finish_reason={last_finish_reason!r}). "
        "Try a specific available OpenRouter model instead of 'openrouter/free'."
    )

### Run OpenRouter-powered safe tailoring

In [ ]:
# make sure you have valid API key
tailoring_result = improve_and_reanalyze_resume(
    JOB_DESCRIPTION, RESUME_TEXT, rewrite_with_your_llm
)
display_before_after_comparison(tailoring_result)
download_tailored_resume(tailoring_result)

### Truthfulness-validator stress test

`validate_resume_claims` is the only thing standing between an LLM's tailored draft and a resume with fabricated skills or metrics. It has not been evaluated on its own; the notebook only ever ran it once on a single manual example. This stress test runs the validator deterministically over many resumes with two kinds of synthetic edits, without calling any LLM or API key:

- **Injected hallucination**: append a fabricated ESCO skill and a fabricated numeric claim to a copy of a real resume. The validator must flag it (this measures detection **recall**).
- **Paraphrase only**: reword existing sentences without adding any new fact. The validator must *not* flag it (this measures the **false-positive rate**).

Report both rates in the thesis; a validator with high recall but a high false-positive rate would make every tailored resume "require review," which defeats the point.


In [ ]:
import random as _stress_random


def run_truthfulness_validator_stress_test(
    resume_texts: list[str],
    injection_skills: list[str],
    injection_numeric_claims: list[str],
    seed: int = RANDOM_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Deterministically stress-test `validate_resume_claims` with synthetic edits.

    For each resume: (1) inject a fabricated skill + numeric claim and confirm it is caught,
    (2) paraphrase two common verbs with no new facts and confirm it is *not* flagged.
    Returns (per_case_results, summary_by_case).
    """
    rng = _stress_random.Random(seed)
    rows = []
    for resume in resume_texts:
        injected_skill = rng.choice(injection_skills)
        injected_claim = rng.choice(injection_numeric_claims)
        hallucinated = resume.rstrip() + f"\n\nAdditional skills: {injected_skill}. Delivered {injected_claim} of impact."
        result = validate_resume_claims(resume, hallucinated, None, SKILL_CONFIG)
        detected = (injected_skill in result["new_skills_requiring_confirmation"]) or bool(result["new_numeric_claims_requiring_confirmation"])
        rows.append({"case": "injected_hallucination", "detected": detected})

        paraphrased = resume.replace("Developed", "Built").replace("Implemented", "Delivered")
        result_paraphrase = validate_resume_claims(resume, paraphrased, None, SKILL_CONFIG)
        false_positive = not result_paraphrase["is_safe_to_show"]
        rows.append({"case": "paraphrase_only", "detected": false_positive})

    results = pd.DataFrame(rows)
    summary = results.groupby("case")["detected"].agg(["mean", "count"]).rename(
        columns={"mean": "flag_rate", "count": "n_trials"}
    )
    summary = summary.rename(index={"injected_hallucination": "injected_hallucination (flag_rate = recall)"})
    summary = summary.rename(index={"paraphrase_only": "paraphrase_only (flag_rate = false_positive_rate)"})
    return results, summary


# Uses the resumes already loaded in Part 1 if available, otherwise falls back to the single
# manual RESUME_TEXT sample from Part 3 repeated. Replace with a larger real sample for the thesis.
_stress_test_resumes = (
    all_resumes["resume_text"].dropna().sample(n=min(30, len(all_resumes)), random_state=RANDOM_SEED).tolist()
    if "all_resumes" in globals() else [RESUME_TEXT] * 30
)

stress_test_results, stress_test_summary = run_truthfulness_validator_stress_test(
    _stress_test_resumes,
    injection_skills=["kubernetes", "tensorflow", "figma", "salesforce", "docker"],
    injection_numeric_claims=["45%", "$2M", "3 years", "120 users"],
)
print(stress_test_summary)
stress_test_results.to_csv("truthfulness_validator_stress_test.csv", index=False)


### Research Module 1 — Run ontology-weighted hybrid ranking

Run this cell after Parts 1–3. It uses the existing ESCO files and existing ranking metrics; it does not retrain or replace SBERT.

In [ ]:
from collections import defaultdict
from typing import Callable, Mapping


@dataclass(frozen=True)
class ResearchExperimentConfig:
    essential_weight: float = 2.0
    optional_weight: float = 1.0
    specificity_bonus: float = 0.10
    occupation_match_threshold: float = 0.05
    validation_fraction: float = 0.20
    alpha_grid: tuple[float, ...] = tuple(np.round(np.arange(0.0, 1.01, 0.1), 1))


RESEARCH_CONFIG = ResearchExperimentConfig()


def _research_data_path(filename: str) -> Path:
    candidates = (Path('/colab/data'), Path('colab/data'), Path('data'))
    path = next((base / filename for base in candidates if (base / filename).exists()), None)
    if path is None:
        raise FileNotFoundError(f'ESCO research data file not found: {filename}')
    return path


def _research_normalize_label(label: object) -> str:
    return ' '.join(str(label).lower().replace('##', '').split())


def load_esco_research_index() -> dict[str, object]:
    """Load ESCO lookup tables used by the three research experiments."""
    skills = pd.read_csv(
        _research_data_path('skills_en.csv'),
        usecols=['conceptUri', 'preferredLabel', 'altLabels', 'hiddenLabels'],
    )
    occupations = pd.read_csv(
        _research_data_path('occupations_en.csv'),
        usecols=['conceptUri', 'preferredLabel', 'altLabels'],
    )
    relations = pd.read_csv(_research_data_path('occupationSkillRelations_en.csv'))
    hierarchy = pd.read_csv(_research_data_path('skillsHierarchy_en.csv'))

    label_rows = []
    for column in ('preferredLabel', 'altLabels', 'hiddenLabels'):
        expanded = skills[['conceptUri', 'preferredLabel']].copy()
        expanded['label'] = skills[column].fillna('').str.split('\n')
        label_rows.append(expanded.explode('label')[['conceptUri', 'preferredLabel', 'label']])
    labels = pd.concat(label_rows, ignore_index=True)
    labels['label'] = labels['label'].map(_research_normalize_label)
    labels = labels.loc[labels['label'].str.len() >= 3].drop_duplicates()
    label_to_uris = labels.groupby('label')['conceptUri'].agg(lambda values: set(values))
    label_to_uri = {label: next(iter(uris)) for label, uris in label_to_uris.items() if len(uris) == 1}
    preferred_by_uri = skills.set_index('conceptUri')['preferredLabel'].map(_research_normalize_label).to_dict()

    hierarchy_depth = {}
    for level in range(4):
        column = f'Level {level} preferred term'
        if column in hierarchy:
            for label in hierarchy[column].dropna().map(_research_normalize_label):
                hierarchy_depth[label] = max(hierarchy_depth.get(label, 0), level)

    occupation_labels = occupations[['conceptUri', 'preferredLabel', 'altLabels']].copy()
    occupation_labels['search_text'] = (
        occupation_labels['preferredLabel'].fillna('') + ' ' + occupation_labels['altLabels'].fillna('').str.replace('\n', ' ', regex=False)
    ).map(_research_normalize_label)
    return {
        'label_to_uri': label_to_uri,
        'preferred_by_uri': preferred_by_uri,
        'hierarchy_depth': hierarchy_depth,
        'relations': relations,
        'occupation_labels': occupation_labels,
    }


def infer_esco_occupation_uri(job_title: str, esco_index: Mapping[str, object], threshold: float = RESEARCH_CONFIG.occupation_match_threshold) -> str | None:
    """Lexically align a posting title to the closest ESCO occupation for weighting only."""
    occupations = esco_index['occupation_labels']
    corpus = occupations['search_text'].tolist() + [_research_normalize_label(job_title)]
    similarities = cosine_similarity(TfidfVectorizer(stop_words='english', ngram_range=(1, 2)).fit_transform(corpus))[-1, :-1]
    best_index = int(np.argmax(similarities))
    return occupations.iloc[best_index]['conceptUri'] if similarities[best_index] >= threshold else None


def calculate_weighted_skill_match_score(job_skills: list[str], resume_skills: list[str], occupation_uri: str | None, esco_index: Mapping[str, object], config: ResearchExperimentConfig = RESEARCH_CONFIG) -> float | None:
    """Calculate ESCO relation- and hierarchy-weighted skill coverage on a 0–100 scale."""
    normalized_job = list(dict.fromkeys(_research_normalize_label(skill) for skill in job_skills if str(skill).strip()))
    normalized_resume = {_research_normalize_label(skill) for skill in resume_skills if str(skill).strip()}
    if not normalized_job:
        return None
    relations = esco_index['relations']
    relation_lookup = {}
    if occupation_uri is not None:
        occupation_relations = relations.loc[relations['occupationUri'] == occupation_uri]
        relation_lookup = occupation_relations.groupby('skillUri')['relationType'].agg(lambda values: set(values)).to_dict()
    weights, matched_weight = [], 0.0
    for skill in normalized_job:
        skill_uri = esco_index['label_to_uri'].get(skill)
        relation_types = relation_lookup.get(skill_uri, set())
        relation_weight = config.essential_weight if 'essential' in relation_types else config.optional_weight
        depth = esco_index['hierarchy_depth'].get(skill, 0)
        weight = relation_weight * (1 + config.specificity_bonus * depth)
        weights.append(weight)
        if skill in normalized_resume:
            matched_weight += weight
    return round(100 * matched_weight / sum(weights), 1) if weights else None


def build_weighted_skill_matrix(jobs: pd.DataFrame, resumes: pd.DataFrame, esco_index: Mapping[str, object], skill_extractor: Callable[[str], list[str]] = extract_onet_skills) -> np.ndarray:
    """Build a job-by-resume weighted ESCO coverage matrix using existing skill extraction."""
    job_skills = [normalize_skills(skill_extractor(text)) for text in jobs['job_description']]
    resume_skills = [normalize_skills(skill_extractor(text)) for text in resumes['resume_text']]
    occupation_uris = [infer_esco_occupation_uri(title, esco_index) for title in jobs['title']]
    matrix = np.zeros((len(jobs), len(resumes)), dtype=float)
    for job_index, skills in enumerate(job_skills):
        for resume_index, resume_skill_set in enumerate(resume_skills):
            matrix[job_index, resume_index] = calculate_weighted_skill_match_score(
                skills, resume_skill_set, occupation_uris[job_index], esco_index
            ) or 0.0
    return matrix


def fuse_semantic_and_weighted_scores(semantic_matrix: np.ndarray, weighted_skill_matrix: np.ndarray, alpha: float) -> np.ndarray:
    """Fuse cosine similarity and weighted skill coverage; output remains a ranking matrix."""
    semantic_0_100 = np.clip(semantic_matrix, 0, 1) * 100
    return alpha * semantic_0_100 + (1 - alpha) * weighted_skill_matrix


def select_hybrid_alpha(semantic_matrix: np.ndarray, weighted_skill_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, config: ResearchExperimentConfig = RESEARCH_CONFIG) -> tuple[float, pd.DataFrame]:
    """Select alpha by NDCG on a validation split; reserve a held-out test split for reporting."""
    rows = []
    for alpha in config.alpha_grid:
        metrics = calculate_ranking_metrics(fuse_semantic_and_weighted_scores(semantic_matrix, weighted_skill_matrix, alpha), jobs, resumes, CONFIG.top_k)
        rows.append({'alpha': alpha, f'NDCG@{CONFIG.top_k}': metrics[f'NDCG@{CONFIG.top_k}'], f'Precision@{CONFIG.top_k}': metrics[f'Precision@{CONFIG.top_k}']})
    results = pd.DataFrame(rows)
    return float(results.sort_values([f'NDCG@{CONFIG.top_k}', 'alpha'], ascending=[False, False]).iloc[0]['alpha']), results


def split_research_jobs(jobs: pd.DataFrame, validation_fraction: float = RESEARCH_CONFIG.validation_fraction) -> tuple[np.ndarray, np.ndarray]:
    """Create deterministic validation/test job indices for selecting alpha without test leakage."""
    if len(jobs) < 5:
        raise ValueError('RQ1 requires at least five jobs for validation/test separation')
    indices = np.random.default_rng(RANDOM_SEED).permutation(len(jobs))
    validation_size = min(max(1, int(round(len(jobs) * validation_fraction))), len(jobs) - 1)
    return indices[:validation_size], indices[validation_size:]


def run_ontology_weighted_hybrid_experiment(jobs: pd.DataFrame, resumes: pd.DataFrame, semantic_matrix: np.ndarray, esco_index: Mapping[str, object] | None = None) -> dict[str, object]:
    """Run RQ1 with validation-selected alpha and held-out test metrics."""
    esco_index = esco_index or load_esco_research_index()
    weighted_matrix = build_weighted_skill_matrix(jobs, resumes, esco_index)
    validation_indices, test_indices = split_research_jobs(jobs)
    validation_jobs = jobs.iloc[validation_indices].reset_index(drop=True)
    test_jobs = jobs.iloc[test_indices].reset_index(drop=True)
    alpha, validation_results = select_hybrid_alpha(semantic_matrix[validation_indices], weighted_matrix[validation_indices], validation_jobs, resumes)
    fused_matrix = fuse_semantic_and_weighted_scores(semantic_matrix, weighted_matrix, alpha)
    metrics = evaluate_similarity_model('ESCO-weighted hybrid', fused_matrix[test_indices], test_jobs, resumes, CONFIG.top_k)
    return {'alpha': alpha, 'validation_results': validation_results, 'validation_indices': validation_indices, 'test_indices': test_indices, 'weighted_skill_matrix': weighted_matrix, 'fused_matrix': fused_matrix, 'metrics': metrics}


### Research Module 2 — Run counterfactual faithfulness evaluation

The experiment accepts a score function, so it can evaluate the current explainable score or the RQ1 hybrid score without duplicating model code.

This counterfactual protocol operationalizes explanation **faithfulness**: whether a score actually depends on the evidence it claims to, not just whether it is plausible. `remove_relevant` tests *comprehensiveness* (removing evidence the score cites should lower the score) and `replace_with_alias`/`insert_unrelated` test *sufficiency/invariance* (paraphrasing cited evidence or adding unrelated content should not move the score). These correspond to the comprehensiveness and sufficiency criteria from the ERASER benchmark (DeYoung et al., 2020) and the faithfulness framing of Jacovi & Goldberg (2020, "Towards Faithfully Interpretable NLP Systems"). Citing this connects RQ2 to established explainability evaluation methodology rather than presenting it as an ad hoc metric.


In [ ]:
def _skill_boundary_pattern(skill: str) -> re.Pattern:
    return re.compile(rf'(?<![A-Za-z0-9]){re.escape(_research_normalize_label(skill))}(?![A-Za-z0-9])', re.IGNORECASE)


def build_esco_alias_index(esco_index: Mapping[str, object]) -> dict[str, list[str]]:
    """Create canonical-skill to alternative-label lookup for alias-invariance tests."""
    by_uri = defaultdict(list)
    for label, uri in esco_index['label_to_uri'].items():
        by_uri[uri].append(label)
    alias_index = {}
    for labels in by_uri.values():
        unique_labels = sorted(set(labels), key=len)
        for label in unique_labels:
            alias_index[label] = [candidate for candidate in unique_labels if candidate != label]
    return alias_index


def create_skill_counterfactual(resume_text: str, skill: str, intervention: str, alias_index: Mapping[str, list[str]], unrelated_skill: str | None = None) -> str:
    """Create deterministic text-only counterfactuals without inventing employment history."""
    normalized_skill = _research_normalize_label(skill)
    pattern = _skill_boundary_pattern(normalized_skill)
    if intervention == 'remove_relevant':
        return re.sub(r'\s{2,}', ' ', pattern.sub('', resume_text)).strip()
    if intervention == 'replace_with_alias':
        alias = next((item for item in alias_index.get(normalized_skill, []) if item != normalized_skill), None)
        return pattern.sub(alias, resume_text) if alias else resume_text
    if intervention == 'insert_unrelated':
        if not unrelated_skill:
            raise ValueError('unrelated_skill is required for insert_unrelated')
        return f'{resume_text.rstrip()}\n\nSkills: {unrelated_skill}'
    raise ValueError('intervention must be remove_relevant, replace_with_alias, or insert_unrelated')


def run_counterfactual_faithfulness_experiment(jobs: pd.DataFrame, resumes: pd.DataFrame, score_pair: Callable[[str, str], float], esco_index: Mapping[str, object] | None = None, max_pairs: int = 30) -> pd.DataFrame:
    """Run RQ2 counterfactual tests using any deterministic JobLens score function."""
    esco_index = esco_index or load_esco_research_index()
    alias_index = build_esco_alias_index(esco_index)
    unrelated_candidates = sorted(esco_index['label_to_uri'])
    records = []
    pair_count = 0
    for _, job in jobs.iterrows():
        for _, resume in resumes.iterrows():
            if pair_count >= max_pairs:
                return pd.DataFrame(records)
            job_skills = normalize_skills(extract_onet_skills(job['job_description']))
            resume_skills = normalize_skills(extract_onet_skills(resume['resume_text']))
            matched = sorted(set(job_skills) & set(resume_skills), key=len, reverse=True)
            if not matched:
                continue
            relevant_skill = matched[0]
            unrelated_skill = next((candidate for candidate in unrelated_candidates if candidate not in set(job_skills) | set(resume_skills)), None)
            if unrelated_skill is None:
                continue
            original_score = float(score_pair(job['job_description'], resume['resume_text']))
            for intervention in ('remove_relevant', 'replace_with_alias', 'insert_unrelated'):
                counterfactual = create_skill_counterfactual(
                    resume['resume_text'], relevant_skill, intervention, alias_index, unrelated_skill
                )
                changed_score = float(score_pair(job['job_description'], counterfactual))
                delta = changed_score - original_score
                records.append({
                    'job_id': job['id'], 'resume_id': resume['id'], 'relevant_skill': relevant_skill,
                    'intervention': intervention, 'original_score': original_score,
                    'counterfactual_score': changed_score, 'score_delta': delta,
                    'faithful': (delta < 0 if intervention == 'remove_relevant' else abs(delta) <= 2.0),
                })
            pair_count += 1
    return pd.DataFrame(records)


def summarise_faithfulness_results(results: pd.DataFrame) -> pd.DataFrame:
    """Return RQ2 measurable outcomes for reporting and plotting."""
    if results.empty:
        return pd.DataFrame(columns=['intervention', 'pairs', 'mean_score_delta', 'mean_absolute_score_delta', 'faithfulness_pass_rate'])
    return results.groupby('intervention', as_index=False).agg(
        pairs=('faithful', 'size'),
        mean_score_delta=('score_delta', 'mean'),
        mean_absolute_score_delta=('score_delta', lambda values: float(np.mean(np.abs(values)))),
        faithfulness_pass_rate=('faithful', 'mean'),
    )


### Research Module 3 — Run uncertainty-aware selective matching

Pass the similarity matrices already produced by the baseline, fine-tuned, and RQ1 hybrid models. The module reports which matches should be routed to human review.

In [ ]:
from sklearn.metrics import roc_auc_score


def calculate_match_uncertainty(similarity_matrices: Mapping[str, np.ndarray]) -> pd.DataFrame:
    """Estimate per-job uncertainty from model disagreement and top-two score margin."""
    if not similarity_matrices:
        raise ValueError('similarity_matrices must contain at least one model matrix')
    matrices = list(similarity_matrices.values())
    shape = matrices[0].shape
    if any(matrix.shape != shape for matrix in matrices):
        raise ValueError('all similarity matrices must have the same job-by-resume shape')
    rows = []
    for job_index in range(shape[0]):
        top_indices, normalized_top_scores, margins = [], [], []
        for matrix in matrices:
            scores = matrix[job_index]
            ranked = np.argsort(scores)[::-1]
            top_indices.append(int(ranked[0]))
            top = float(scores[ranked[0]])
            second = float(scores[ranked[1]]) if len(ranked) > 1 else top
            score_range = float(np.max(scores) - np.min(scores))
            normalized_top_scores.append((top - float(np.min(scores))) / score_range if score_range else 1.0)
            margins.append((top - second) / score_range if score_range else 0.0)
        vote_share = max(top_indices.count(index) for index in set(top_indices)) / len(top_indices)
        disagreement = 1 - vote_share
        uncertainty = float(np.var(normalized_top_scores) + (1 - np.mean(margins)) + disagreement)
        rows.append({'job_index': job_index, 'uncertainty': uncertainty, 'model_disagreement': disagreement, 'mean_top2_margin': float(np.mean(margins))})
    return pd.DataFrame(rows)


def calculate_naive_confidence_baseline(reference_matrix: np.ndarray) -> pd.DataFrame:
    """Single-model top-1/top-2 margin uncertainty, with no model disagreement signal.

    This is the naive baseline RQ3's fused uncertainty measure must beat: if a plain top-1
    similarity margin from one model gives the same error-detection AUROC as the fused,
    multi-model disagreement measure, the extra machinery in `calculate_match_uncertainty`
    is not earning its complexity.
    """
    rows = []
    for job_index in range(reference_matrix.shape[0]):
        scores = reference_matrix[job_index]
        ranked = np.argsort(scores)[::-1]
        top = float(scores[ranked[0]])
        second = float(scores[ranked[1]]) if len(ranked) > 1 else top
        score_range = float(np.max(scores) - np.min(scores))
        margin = (top - second) / score_range if score_range else 0.0
        rows.append({'job_index': job_index, 'uncertainty': 1 - margin})
    return pd.DataFrame(rows)


def evaluate_selective_ranking(reference_matrix: np.ndarray, uncertainty: pd.DataFrame, jobs: pd.DataFrame, resumes: pd.DataFrame, thresholds: list[float] | None = None, top_k: int = CONFIG.top_k) -> tuple[pd.DataFrame, float | None]:
    """Evaluate coverage/risk trade-offs when uncertain top-1 matches are deferred."""
    resume_ids = resumes['id'].tolist()
    records = []
    for row_index, (_, job) in enumerate(jobs.iterrows()):
        ranked = np.argsort(reference_matrix[row_index])[::-1]
        correct_rank = list(ranked).index(resume_ids.index(job['correct_resume_id'])) + 1
        records.append({'job_index': row_index, 'top1_correct': int(correct_rank == 1), f'hit_at_{top_k}': int(correct_rank <= top_k)})
    outcomes = pd.DataFrame(records).merge(uncertainty, on='job_index', validate='one_to_one')
    thresholds = thresholds or sorted(outcomes['uncertainty'].quantile(np.linspace(0, 1, 11)).unique())
    rows = []
    for threshold in thresholds:
        retained = outcomes.loc[outcomes['uncertainty'] <= threshold]
        if retained.empty:
            continue
        rows.append({
            'threshold': float(threshold), 'coverage': len(retained) / len(outcomes),
            'review_rate': 1 - len(retained) / len(outcomes),
            'selective_top1_accuracy': float(retained['top1_correct'].mean()),
            f'selective_Precision@{top_k}': float(retained[f'hit_at_{top_k}'].mean()),
        })
    error_labels = 1 - outcomes['top1_correct']
    auroc = float(roc_auc_score(error_labels, outcomes['uncertainty'])) if error_labels.nunique() == 2 else None
    return pd.DataFrame(rows), auroc


def select_review_threshold(selective_results: pd.DataFrame, minimum_coverage: float = 0.70) -> float:
    """Choose the most accurate threshold that still automatically handles the requested coverage."""
    eligible = selective_results.loc[selective_results['coverage'] >= minimum_coverage]
    if eligible.empty:
        raise ValueError('no threshold satisfies minimum_coverage')
    return float(eligible.sort_values('selective_top1_accuracy', ascending=False).iloc[0]['threshold'])


def build_research_similarity_matrices(models: Mapping[str, SentenceTransformer], resumes: pd.DataFrame, jobs: pd.DataFrame, batch_size: int = CONFIG.batch_size) -> dict[str, np.ndarray]:
    """Create aligned matrices for RQ3 from existing SentenceTransformer models."""
    return {
        name: build_embedding_similarity_matrix(model, resumes, jobs, batch_size)
        for name, model in models.items()
    }


def make_current_match_score_function(ner_model: Any, similarity_model: SentenceTransformer, skill_config: SkillExtractionConfig = SKILL_CONFIG, analysis_config: ApplicationAnalysisConfig = APPLICATION_CONFIG) -> Callable[[str, str], float]:
    """Adapt the current explainable score for the RQ2 counterfactual experiment."""
    def score_pair(job_description: str, resume_text: str) -> float:
        analysis = analyze_application(
            job_description, resume_text, skill_config=skill_config,
            analysis_config=analysis_config, ner_model=ner_model, similarity_model=similarity_model,
        )
        return float(analysis['match_score'])
    return score_pair


## Run All Research Experiments

Set `RUN_RESEARCH_EXPERIMENTS` to `True` in the final cell to run RQ1–RQ3 and save reproducible CSV artefacts. The cell reuses the existing datasets, fine-tuned SBERT model, ESCO data, and skill-extraction model.

In [ ]:
# Change to True only when you are ready to run all three research experiments.
RUN_RESEARCH_EXPERIMENTS = False
RESEARCH_MAX_COUNTERFACTUAL_PAIRS = 15
RESEARCH_OUTPUT_DIR = Path('joblens_research_results')


def run_all_research_experiments(config: PipelineConfig = CONFIG) -> dict[str, object]:
    """Run RQ1-RQ3 with existing JobLens components and save all reportable artefacts.

    Every RQ1-3 evaluation below runs on the held-out `test_jobs`/`test_resumes` split, the same
    split used for the Part 1 comparison table; `train_jobs`/`train_resumes` is used only to
    fine-tune SBERT. This keeps the research experiments consistent with the leakage-free
    evaluation protocol used in Part 1.
    """
    RESEARCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('1/7 Loading and preparing train/test-disjoint evaluation data')
    train_resumes, train_jobs, test_resumes, test_jobs = prepare_evaluation_data(
        load_resume_dataset(), load_job_postings(), config
    )
    print(f'   train: resumes={len(train_resumes)}, jobs={len(train_jobs)}')
    print(f'   test:  resumes={len(test_resumes)}, jobs={len(test_jobs)}')

    print('2/7 Loading the fine-tuned SBERT model (trained on the train split only) and ESCO index')
    finetuned_model, model_source = load_or_train_finetuned_model(train_resumes, train_jobs, config)
    esco_index = load_esco_research_index()

    print('3/7 Building pretrained and fine-tuned similarity matrices on the held-out test split')
    pretrained_model = SentenceTransformer(config.base_model_name, device=config.device)
    pretrained_matrix = build_embedding_similarity_matrix(pretrained_model, test_resumes, test_jobs, config.batch_size)
    finetuned_matrix = build_embedding_similarity_matrix(finetuned_model, test_resumes, test_jobs, config.batch_size)

    print('4/7 Running RQ1: ESCO ontology-weighted hybrid ranking')
    rq1 = run_ontology_weighted_hybrid_experiment(test_jobs, test_resumes, finetuned_matrix, esco_index)
    rq1_ndcg_ci = bootstrap_ci(rq1['metrics']['ndcg_array'], config.n_bootstrap)
    rq1_precision_ci = bootstrap_ci(rq1['metrics']['hit_array'], config.n_bootstrap)
    rq1['validation_results'].to_csv(RESEARCH_OUTPUT_DIR / 'rq1_alpha_validation.csv', index=False)
    pd.DataFrame([{key: value for key, value in rq1['metrics'].items() if key not in ('details', 'hit_array', 'ndcg_array')}]).to_csv(
        RESEARCH_OUTPUT_DIR / 'rq1_test_metrics.csv', index=False
    )

    print('5/7 Running RQ2: counterfactual explanation-faithfulness evaluation')
    ner_model = load_skill_ner_model(SKILL_CONFIG)
    score_pair = make_current_match_score_function(ner_model, finetuned_model)
    counterfactual_results = run_counterfactual_faithfulness_experiment(
        test_jobs, test_resumes, score_pair, esco_index, max_pairs=RESEARCH_MAX_COUNTERFACTUAL_PAIRS
    )
    faithfulness_summary = summarise_faithfulness_results(counterfactual_results)
    counterfactual_results.to_csv(RESEARCH_OUTPUT_DIR / 'rq2_counterfactual_pairs.csv', index=False)
    faithfulness_summary.to_csv(RESEARCH_OUTPUT_DIR / 'rq2_faithfulness_summary.csv', index=False)

    print('6/7 Running RQ3: uncertainty-aware selective matching')
    model_matrices = {
        'pretrained_sbert': pretrained_matrix,
        'finetuned_sbert': finetuned_matrix,
        'esco_weighted_hybrid': rq1['fused_matrix'],
    }
    uncertainty = calculate_match_uncertainty(model_matrices)
    selective_results, error_detection_auroc = evaluate_selective_ranking(
        rq1['fused_matrix'], uncertainty, test_jobs, test_resumes, top_k=config.top_k
    )
    naive_uncertainty = calculate_naive_confidence_baseline(rq1['fused_matrix'])
    _, naive_error_detection_auroc = evaluate_selective_ranking(
        rq1['fused_matrix'], naive_uncertainty, test_jobs, test_resumes, top_k=config.top_k
    )
    review_threshold = select_review_threshold(selective_results)
    uncertainty.to_csv(RESEARCH_OUTPUT_DIR / 'rq3_job_uncertainty.csv', index=False)
    selective_results.to_csv(RESEARCH_OUTPUT_DIR / 'rq3_risk_coverage.csv', index=False)

    print('7/7 Saving the reproducible experiment summary')
    summary = {
        'fine_tuned_model_source': model_source,
        'rq1_selected_alpha': rq1['alpha'],
        'rq1_test_ndcg': round(rq1_ndcg_ci[0], 3),
        'rq1_test_ndcg_ci': f'[{rq1_ndcg_ci[1]:.3f}, {rq1_ndcg_ci[2]:.3f}]',
        'rq1_test_precision': round(rq1_precision_ci[0], 3),
        'rq1_test_precision_ci': f'[{rq1_precision_ci[1]:.3f}, {rq1_precision_ci[2]:.3f}]',
        'rq2_pairs': len(counterfactual_results),
        'rq3_error_detection_auroc_fused': error_detection_auroc,
        'rq3_error_detection_auroc_naive_baseline': naive_error_detection_auroc,
        'rq3_review_threshold': review_threshold,
    }
    pd.DataFrame([summary]).to_csv(RESEARCH_OUTPUT_DIR / 'research_experiment_summary.csv', index=False)
    print('\n Research experiments complete')
    print(pd.DataFrame([summary]).to_string(index=False))
    print(f'Artifacts saved to: {RESEARCH_OUTPUT_DIR.resolve()}')
    return {
        'summary': summary, 'rq1': rq1, 'rq2_pairs': counterfactual_results,
        'rq2_summary': faithfulness_summary, 'rq3_uncertainty': uncertainty,
        'rq3_selective_results': selective_results,
    }


if RUN_RESEARCH_EXPERIMENTS:
    research_results = run_all_research_experiments()
else:
    print('Research experiments are disabled. Set RUN_RESEARCH_EXPERIMENTS = True to run RQ1–RQ3.')
